# TransFit Quick Tutorial

This notebook shows the shortest useful workflow: build a model, draw forward light curves, run a small demonstration fit, and save the result. The sampling settings below are intentionally short for a tutorial; use longer chains for science runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import transfit as tf
from transfit.modules.sed import CutoffBlackbodySED

ROOT = Path.cwd()
if not (ROOT / "examples" / "data" / "sn1993j_lbol.txt").exists():
    ROOT = ROOT.parent

## 1. Choose a model

Use `model_param_names` or `param_template` to inspect the required parameter names.

In [ ]:
tf.model_param_names("nickel")

In [ ]:
params = {
    "M_ej": 3.0,
    "v_ej": 1.0,
    "E_Th_in": 1.5,
    "M_Ni": 0.08,
    "R_0": 120.0,
    "x_Ni": 0.2,
    "kappa": 0.12,
    "kappa_gamma": 0.03,
    "T_floor": 4500.0,
}

## 2. Bolometric forward model

In [ ]:
bol = tf.lightcurve_bol(
    model="nickel",
    params=params,
    z=0.001728,
    t_max_days=120.0,
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bol.t_days, bol.Lbol, color="black")
ax.set_yscale("log")
ax.set_xlabel("Observer-frame time (days)")
ax.set_ylabel("Bolometric luminosity (erg s$^{-1}$)")
fig.tight_layout()

## 3. Multi-band forward model

In [ ]:
filters = {
    "B": "johnson_cousins.B",
    "V": "johnson_cousins.V",
    "R": "johnson_cousins.R",
    "I": "johnson_cousins.I",
}

mb = tf.lightcurve_multiband(
    model="nickel",
    params=params,
    z=0.001728,
    distance_modulus=29.84,
    filters=filters,
    bands=["B", "V", "R", "I"],
    y_kind="mag",
    mag_system="vega",
    t_max_days=120.0,
)

fig, ax = plt.subplots(figsize=(6, 4))
for band in mb.bands:
    ax.plot(mb.t_days, mb.y[band], label=band)
ax.invert_yaxis()
ax.set_xlabel("Observer-frame time (days)")
ax.set_ylabel("Vega magnitude")
ax.legend()
fig.tight_layout()

## 4. A short bolometric fit

This cell demonstrates the API. The chain is deliberately short, so do not use these sampler settings for publication results.

In [ ]:
arr = np.loadtxt(ROOT / "examples" / "data" / "sn1993j_lbol.txt")
data = tf.BolometricData(
    t_days=arr[:, 0] - arr[:, 0].min(),
    y=arr[:, 1],
    yerr=arr[:, 2],
)

res = tf.fit_bol(
    data=data,
    model="nickel",
    z=0.001728,
    priors={
        "M_ej": (0.5, 8.0),
        "v_ej": (0.2, 3.0),
        "E_Th_in": (0.05, 8.0),
        "M_Ni": ("log10", -3.0, -0.2),
        "R_0": (10.0, 400.0),
    },
    fixed={
        "x_Ni": 0.2,
        "kappa": 0.12,
        "kappa_gamma": 0.03,
    },
    sampler_kwargs={
        "nwalkers": 32,
        "nsteps": 80,
        "burnin": 20,
        "seed": 123,
        "progress": False,
        "robust_init": False,
    },
    model_kwargs={"t_max_days": 160.0},
)

res.best_params_raw

## 5. Plot and save the fit

In [ ]:
fig = tf.plot.fit_bol(res, data=data, show_1sigma=True)

outpath = tf.save(res, ROOT / "mcmc_out" / "tutorial_nickel_bol.npz")
loaded = tf.load(outpath)
print(outpath)
print(loaded["samples"].shape)

## 6. Optional: UV cutoff SED

`CutoffBlackbodySED` can be passed to multi-band calls when a blackbody is too bright in the blue/UV.

In [ ]:
sed = CutoffBlackbodySED(
    cutoff_wavelength_A=3000.0,
    uv_slope=2.0,
    min_factor=0.1,
)

mb_cutoff = tf.lightcurve_multiband(
    model="nickel",
    params=params,
    z=0.001728,
    distance_modulus=29.84,
    filters=filters,
    bands=["B", "V"],
    y_kind="mag",
    mag_system="vega",
    sed=sed,
    t_max_days=120.0,
)

fig, ax = plt.subplots(figsize=(6, 4))
for band in mb_cutoff.bands:
    ax.plot(mb_cutoff.t_days, mb_cutoff.y[band], label=f"{band} cutoff")
ax.invert_yaxis()
ax.set_xlabel("Observer-frame time (days)")
ax.set_ylabel("Vega magnitude")
ax.legend()
fig.tight_layout()